<a href="https://colab.research.google.com/github/dovalless/1790-introduccion-a-sql-con-mysql/blob/main/proyecto_detector_de_neumonia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# Configuramos tu llave de acceso
os.environ['KAGGLE_USERNAME'] = "Tdarwinmovallescesar" # <--- ESCRIBE AQUÍ TU NOMBRE DE USUARIO (está en tu perfil de Kaggle)
os.environ['KAGGLE_KEY'] = "7e80c82cb93c002a160226a535b6d5dc"

# Probamos si funciona
!kaggle datasets list -s "chest-xray"


ref                                                             title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------------  -------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
bachrr/covid-chest-xray                                         COVID-19 chest xray                                  252641948  2020-05-15 00:30:50.877000          14388        254  0.9411765        
nikhilpandey360/chest-xray-masks-and-labels                     Chest Xray Masks and Labels                        10281955076  2019-01-21 09:11:43.557000          24899        214  0.75             
alifrahman/chestxraydataset                                     chest-xray-dataset                                  1222937573  2020-08-31 19:36:35.387000           4105         35  0.625            


In [2]:
import psutil

# Verificar espacio en disco
obj_disk = psutil.disk_usage('/')
print(f"Total: {obj_disk.total / (1024**3):.2f} GB")
print(f"Usado: {obj_disk.used / (1024**3):.2f} GB")
print(f"Libre: {obj_disk.free / (1024**3):.2f} GB")

Total: 112.64 GB
Usado: 43.40 GB
Libre: 69.22 GB


In [3]:
# Descarga rápida
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip -d /content/dataset

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [00:21<00:00, 198MB/s]
100% 2.29G/2.29G [00:21<00:00, 117MB/s]


In [4]:
# Descomprimir forzando el reemplazo (la opción -o lo hace automático)
!unzip -o -q chest-xray-pneumonia.zip -d /content/dataset



In [5]:
# Instala la librería necesaria (tarda unos 30 segundos)
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.6 MB/s eta 0:00:00


In [6]:
from ultralytics import YOLO

# 1. Cargar modelo base (5MB aprox)
model = YOLO("yolo11n-cls.pt")

# 2. Entrenar (imgsz=224 para que vuele en Windows)
# Usamos la carpeta 'chest_xray' que crea el unzip
model.train(
    data="/content/dataset/chest_xray",
    epochs=10,
    imgsz=224,
    batch=32
)

# 3. EXPORTAR PARA WINDOWS (Formato OpenVINO)
# Este paso es el que hace que funcione en PCs de bajos recursos
model.export(format="openvino")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/chest_xray, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8

'/content/runs/classify/train/weights/best_openvino_model'

In [ ]:
from ultralytics import YOLO

# 1. CARGAR MODELO NANO DE CLASIFICACIÓN (El más ligero: 5MB)
# Usamos 'yolo11n-cls.pt' para que identifique la enfermedad directamente
model = YOLO("yolo11n-cls.pt")

# 2. INICIAR EL ENTRENAMIENTO
# Importante: Como es clasificación, pasamos la RUTA de la carpeta, no el .yaml
results = model.train(
    data="/content/dataset/chest_xray", # La carpeta que descomprimiste
    epochs=10,
    imgsz=224,      # Mantiene la velocidad en tu PC vieja
    batch=32,       # Aprovecha la RAM de Colab
    name="modelo_medico_vivos"
)

# 3. EXPORTAR A OPENVINO (La "magia" para Windows)
# Este paso crea la carpeta que descargarás para tu PC
model.export(format="openvino")


Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/chest_xray, degrees=0.0, deterministic=True, device=, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=modelo_medico_vivos, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

In [7]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# Ruta donde YOLO guarda los resultados
results_dir = '/content/runs/detect/chest_xray_model'
results_png = os.path.join(results_dir, 'results.png')

if os.path.exists(results_png):
    plt.figure(figsize=(12, 8))
    img = mpimg.imread(results_png)
    plt.imshow(img)
    plt.axis('off')
    plt.title('Progreso del Entrenamiento (Métricas y Pérdida)')
    plt.show()
else:
    print('Las gráficas se generarán cuando termine al menos la primera época.')
    # Listar archivos actuales para ver qué se ha generado
    if os.path.exists(results_dir):
        print('Archivos generados hasta ahora:', os.listdir(results_dir))

Las gráficas se generarán cuando termine al menos la primera época.


### Probar el modelo OpenVINO exportado

Para probar el modelo, primero necesitas cargar el modelo OpenVINO que se exportó. Luego, puedes usarlo para hacer predicciones en nuevas imágenes.

In [8]:
from ultralytics import YOLO
import cv2
import numpy as np
import os

# La ruta al modelo exportado en OpenVINO
exported_model_path = '/content/runs/classify/train/weights/best_openvino_model/'

# Verificar si la ruta existe y contiene el archivo .xml del modelo OpenVINO
if not os.path.exists(exported_model_path):
    print(f"Error: La ruta del modelo exportado no existe: {exported_model_path}")
else:
    # Cargar el modelo YOLO con el formato OpenVINO
    # Asegúrate de especificar el formato 'openvino' si no lo carga automáticamente
    try:
        model = YOLO(exported_model_path)
        print("Modelo OpenVINO cargado exitosamente.")

        # Ejemplo de una imagen para predecir (usaremos una del dataset de prueba)
        # Asegúrate de que esta ruta sea válida para tu dataset
        # Cambiado a una imagen de PNEUMONIA
        test_image_path = '/content/dataset/chest_xray/test/PNEUMONIA/person152_bacteria_724.jpeg'

        if os.path.exists(test_image_path):
            # Realizar una predicción
            results = model.predict(test_image_path)

            # Mostrar los resultados
            for r in results:
                # Los resultados de clasificación tienen la propiedad `probs`
                if hasattr(r, 'probs'):
                    print("Probabilidades de clasificación:")
                    # r.names contiene los nombres de las clases
                    for i, prob in enumerate(r.probs.data):
                        class_name = model.names[i]
                        print(f"  {class_name}: {prob:.4f}")
                else:
                    print(r.tojson())

            # Opcional: mostrar la imagen con el resultado si el modelo lo visualiza
            # En clasificación, usualmente no hay bbox que mostrar, pero la librería puede generar una imagen con texto
            # if hasattr(r, 'plot'):
            #     im_bgr = r.plot()  # plot a BGR numpy array of predictions
            #     im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)  # convert to RGB
            #     plt.imshow(im_rgb)
            #     plt.axis('off')
            #     plt.show()

        else:
            print(f"Error: La imagen de prueba no existe: {test_image_path}")

    except Exception as e:
        print(f"Ocurrió un error al cargar o usar el modelo: {e}")

Modelo OpenVINO cargado exitosamente.
Loading /content/runs/classify/train/weights/best_openvino_model/ for OpenVINO inference...
Using OpenVINO LATENCY mode for batch=1 inference on CPU...

image 1/1 /content/dataset/chest_xray/test/PNEUMONIA/person152_bacteria_724.jpeg: 224x224 PNEUMONIA 0.91, NORMAL 0.09, 10.2ms
Speed: 12.0ms preprocess, 10.2ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
Probabilidades de clasificación:
  NORMAL: 0.0943
  PNEUMONIA: 0.9057


In [9]:
import os

# Ruta de la carpeta de prueba para PNEUMONIA
pneumonia_test_dir = '/content/dataset/chest_xray/test/PNEUMONIA/'

if os.path.exists(pneumonia_test_dir):
    print(f"Contenido de {pneumonia_test_dir}:")
    files = os.listdir(pneumonia_test_dir)
    # Imprimir los primeros 10 archivos si hay muchos
    for f in files[:10]:
        print(f)
    if len(files) > 10:
        print(f"...y {len(files) - 10} archivos más.")
    if not files:
        print("La carpeta está vacía.")
else:
    print(f"Error: El directorio no existe: {pneumonia_test_dir}")

Contenido de /content/dataset/chest_xray/test/PNEUMONIA/:
person111_bacteria_533.jpeg
person114_bacteria_546.jpeg
person102_bacteria_487.jpeg
person137_bacteria_655.jpeg
person71_virus_131.jpeg
person1_virus_9.jpeg
person83_bacteria_409.jpeg
person91_bacteria_446.jpeg
person152_bacteria_721.jpeg
person128_bacteria_606.jpeg
...y 380 archivos más.


In [10]:
from ultralytics import YOLO
import os

# La ruta al modelo exportado en OpenVINO
exported_model_path = '/content/runs/classify/train/weights/best_openvino_model/'

# Ruta de la carpeta de prueba para PNEUMONIA
pneumonia_test_dir = '/content/dataset/chest_xray/test/PNEUMONIA/'

# Verificar si la ruta del modelo y la carpeta de imágenes existen
if not os.path.exists(exported_model_path):
    print(f"Error: La ruta del modelo exportado no existe: {exported_model_path}")
elif not os.path.exists(pneumonia_test_dir):
    print(f"Error: El directorio de imágenes de prueba no existe: {pneumonia_test_dir}")
else:
    try:
        # Cargar el modelo una sola vez
        model = YOLO(exported_model_path)
        print("Modelo OpenVINO cargado exitosamente.\n")

        files = os.listdir(pneumonia_test_dir)
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

        if not image_files:
            print("La carpeta de PNEUMONIA no contiene imágenes.\n")
        else:
            print(f"Realizando predicciones para {len(image_files)} imágenes de PNEUMONIA...")
            for i, image_name in enumerate(image_files):
                test_image_path = os.path.join(pneumonia_test_dir, image_name)
                print(f"\n--- Prediciendo para: {image_name} ({i+1}/{len(image_files)}) ---")

                results = model.predict(test_image_path)

                for r in results:
                    if hasattr(r, 'probs'):
                        print("  Probabilidades de clasificación:")
                        for j, prob in enumerate(r.probs.data):
                            class_name = model.names[j]
                            print(f"    {class_name}: {prob:.4f}")
                    else:
                        print(r.tojson())

    except Exception as e:
        print(f"Ocurrió un error al cargar o usar el modelo: {e}")


Modelo OpenVINO cargado exitosamente.

Realizando predicciones para 390 imágenes de PNEUMONIA...

--- Prediciendo para: person111_bacteria_533.jpeg (1/390) ---
Loading /content/runs/classify/train/weights/best_openvino_model/ for OpenVINO inference...
Using OpenVINO LATENCY mode for batch=1 inference on CPU...

image 1/1 /content/dataset/chest_xray/test/PNEUMONIA/person111_bacteria_533.jpeg: 224x224 PNEUMONIA 1.00, NORMAL 0.00, 10.1ms
Speed: 6.5ms preprocess, 10.1ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)
  Probabilidades de clasificación:
    NORMAL: 0.0000
    PNEUMONIA: 1.0000

--- Prediciendo para: person114_bacteria_546.jpeg (2/390) ---

image 1/1 /content/dataset/chest_xray/test/PNEUMONIA/person114_bacteria_546.jpeg: 224x224 PNEUMONIA 1.00, NORMAL 0.00, 8.9ms
Speed: 7.1ms preprocess, 8.9ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)
  Probabilidades de clasificación:
    NORMAL: 0.0000
    PNEUMONIA: 1.0000

--- Prediciendo para: pers

KeyboardInterrupt: 

In [15]:
import os
from google.colab import files

# Ruta del modelo OpenVINO exportado
exported_model_dir = '/content/runs/classify/train/weights/best_openvino_model/'
output_zip_name = 'best_openvino_model.zip'

if os.path.exists(exported_model_dir):
    # Comprimir la carpeta del modelo
    !zip -r $output_zip_name $exported_model_dir
    print(f"\nModelo OpenVINO comprimido en: {output_zip_name}")

    # Descargar el archivo zip
    # files.download(output_zip_name)
    print("Puedes descargar el archivo usando el icono de la carpeta a la izquierda, o el siguiente comando:")
    print(f"!cp {output_zip_name} . && ls -lh {output_zip_name}")
    print("Una vez que el archivo .zip esté visible en el panel de archivos de Colab (panel izquierdo), haz clic derecho y selecciona 'Descargar'.")
else:
    print(f"Error: La carpeta del modelo exportado no existe: {exported_model_dir}")


updating: content/runs/classify/train/weights/best_openvino_model/ (stored 0%)
updating: content/runs/classify/train/weights/best_openvino_model/best.bin (deflated 10%)
updating: content/runs/classify/train/weights/best_openvino_model/best.xml (deflated 94%)
updating: content/runs/classify/train/weights/best_openvino_model/metadata.yaml (deflated 36%)

Modelo OpenVINO comprimido en: best_openvino_model.zip
Puedes descargar el archivo usando el icono de la carpeta a la izquierda, o el siguiente comando:
!cp best_openvino_model.zip . && ls -lh best_openvino_model.zip
Una vez que el archivo .zip esté visible en el panel de archivos de Colab (panel izquierdo), haz clic derecho y selecciona 'Descargar'.
